# Chapter 4: Systems of Ordinary Differential Equations

## Systems of ODEs

### Engineering Case Study: Mass-Spring-Damper System

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# System parameters
m1, m2 = 1.0, 2.0
k1, k2 = 2.0, 1.0
c1, c2 = 0.5, 0.3

# Define the system of ODEs
def mass_spring_damper(t, y):
    # y = [x1, dx1/dt, x2, dx2/dt]
    dy = np.zeros(4)

    dy[0] = y[1]  # dx1/dt
    dy[1] = (1/m1) * (-k1*y[0] - (c1+c2)*y[1] + k2*(y[2]-y[0]) + c2*y[3])
    dy[2] = y[3]  # dx2/dt
    dy[3] = (1/m2) * (-k2*(y[2]-y[0]) - c2*(y[3]-y[1]))

    return dy

# Initial conditions: x1(0)=1, dx1/dt(0)=0, x2(0)=0, dx2/dt(0)=0
y0 = [1.0, 0.0, 0.0, 0.0]
t_span = (0, 20)
t_eval = np.linspace(0, 20, 1000)

# Solve the system
sol = solve_ivp(mass_spring_damper, t_span, y0, t_eval=t_eval, method='RK45')

# Plot the results
plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1)
plt.plot(sol.t, sol.y[0], 'b-', label='$x_1(t)$')
plt.plot(sol.t, sol.y[2], 'r-', label='$x_2(t)$')
plt.grid(True)
plt.xlabel('Time (s)')
plt.ylabel('Position (m)')
plt.legend()
plt.title('Position vs Time')

plt.subplot(2, 1, 2)
plt.plot(sol.t, sol.y[1], 'b--', label='$v_1(t)$')
plt.plot(sol.t, sol.y[3], 'r--', label='$v_2(t)$')
plt.grid(True)
plt.xlabel('Time (s)')
plt.ylabel('Velocity (m/s)')
plt.legend()
plt.title('Velocity vs Time')

plt.tight_layout()
plt.show()

### SymPy for Analytical Solutions

In [ ]:
import sympy as sp

# Define symbols
t = sp.Symbol('t')
x = sp.Function('x')(t)
y = sp.Function('y')(t)

# Define differential equations
eq1 = sp.Eq(x.diff(t), 2*x + y)
eq2 = sp.Eq(y.diff(t), x + 2*y)

# Solve the system
system = [eq1, eq2]
solution = sp.dsolve(system)
print("Solution:")
print(solution)

# Apply the initial conditions x(0) = 1, y(0) = 0 by solving
# for the integration constants C1 and C2
C1, C2 = sp.symbols('C1 C2')
constants = sp.solve([solution[0].rhs.subs(t, 0) - 1,
                      solution[1].rhs.subs(t, 0) - 0], [C1, C2])
print("Constants:", constants)
x_sol = solution[0].rhs.subs(constants)
y_sol = solution[1].rhs.subs(constants)

# Convert to lambda functions for plotting
x_func = sp.lambdify(t, x_sol, 'numpy')
y_func = sp.lambdify(t, y_sol, 'numpy')

import numpy as np
import matplotlib.pyplot as plt

# Plot the solution
t_vals = np.linspace(0, 2, 100)
x_vals = x_func(t_vals)
y_vals = y_func(t_vals)

plt.figure(figsize=(10, 6))
plt.plot(t_vals, x_vals, 'b-', label='x(t)')
plt.plot(t_vals, y_vals, 'r-', label='y(t)')
plt.grid(True)
plt.xlabel('Time (t)')
plt.ylabel('Value')
plt.legend()
plt.title('Analytical Solution of the Linear System')
plt.show()

# Phase portrait
plt.figure(figsize=(8, 8))
plt.plot(x_vals, y_vals, 'g-')
plt.grid(True)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Phase Portrait')
plt.axis('equal')
plt.show()

## Theory of ODE Systems

### Phase Space Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Define the ODE system
def damped_oscillator(t, xy, k=1.0, c=0.2):
    x, y = xy
    dxdt = y
    dydt = -k*x - c*y
    return [dxdt, dydt]

# Set up the phase space grid
x = np.linspace(-3, 3, 20)
y = np.linspace(-3, 3, 20)
X, Y = np.meshgrid(x, y)

# Calculate the gradient at each point
U = Y
V = -X - 0.2*Y  # Using k=1, c=0.2

# Normalize the vectors for better visualization
norm = np.sqrt(U**2 + V**2)
U = U/norm
V = V/norm

# Plot the phase portrait
plt.figure(figsize=(10, 10))

# Plot the vector field
plt.quiver(X, Y, U, V, color='gray', alpha=0.5)

# Plot several trajectories
for x0 in [-3, -2, -1, 0, 1, 2, 3]:
    for y0 in [-3, -2, -1, 0, 1, 2, 3]:
        if abs(x0) + abs(y0) > 0:  # Skip the origin
            sol = solve_ivp(damped_oscillator, [0, 10], [x0, y0],
                           dense_output=True, args=(1.0, 0.2))
            t_dense = np.linspace(0, 10, 1000)
            y_dense = sol.sol(t_dense)
            plt.plot(y_dense[0], y_dense[1], '-', linewidth=1)

# Add a special trajectory with arrow
sol = solve_ivp(damped_oscillator, [0, 15], [2, 0],
               dense_output=True, args=(1.0, 0.2))
t_dense = np.linspace(0, 15, 1000)
y_dense = sol.sol(t_dense)
plt.plot(y_dense[0], y_dense[1], 'r-', linewidth=2)
# Add arrows to indicate direction
arrow_idx = [100, 200, 300, 400]
for idx in arrow_idx:
    plt.arrow(y_dense[0][idx], y_dense[1][idx],
              (y_dense[0][idx+1]-y_dense[0][idx])*20,
              (y_dense[1][idx+1]-y_dense[1][idx])*20,
              head_width=0.1, head_length=0.15, fc='r', ec='r')

plt.grid(True)
plt.xlabel('Position (x)')
plt.ylabel('Velocity (y)')
plt.title('Phase Portrait of Damped Harmonic Oscillator')
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
plt.axis('equal')
plt.show()

## Constant-Coefficient Systems

### Solution via Matrix Exponential

In [ ]:
import sympy as sp

# Define symbolic variable and matrix
t = sp.Symbol('t')
A = sp.Matrix([[3, 1], [4, -1]])

# Calculate eigenvalues and eigenvectors
eigenvalues = A.eigenvals()
print("Eigenvalues:", eigenvalues)

P, D = A.diagonalize()
print("P =\n", P)
print("D =\n", D)

# Calculate matrix exponential via diagonalization
exp_D = sp.Matrix([
    [sp.exp(D[0,0]*t), 0],
    [0, sp.exp(D[1,1]*t)]
])
exp_A = sp.simplify(P * exp_D * P.inv())

print("Matrix exponential e^(At) =\n", exp_A)

# Verify that e^(At) satisfies the differential equation
derivative = exp_A.diff(t)
check = sp.simplify(derivative - A*exp_A)
print("Verification (should be zero matrix):\n", check)

## Jordan Canonical Form and Generalized Eigenvectors

### Computing Jordan Form with Python

In [ ]:
import sympy as sp

# Define the matrix
A = sp.Matrix([[3, 1, 0], [0, 3, 1], [0, 0, 3]])

# Compute Jordan form
P, J = A.jordan_form()

print("Original matrix A:")
print(A)
print("\nJordan form J:")
print(J)
print("\nTransformation matrix P:")
print(P)

# Verify: A = P*J*P^{-1}
verification = sp.simplify(P * J * P.inv() - A)
print("\nVerification (should be zero):")
print(verification)

# Now compute the matrix exponential e^{At} = P e^{Jt} P^{-1}
t = sp.Symbol('t')
exp_J = (J * t).exp()  # exponential of the Jordan block
exp_A = sp.simplify(P * exp_J * P.inv())

print("\nMatrix exponential e^(At):")
print(exp_A)

## Nonlinear Systems: Equilibrium Analysis and Linearization

### Python Implementation: Equilibrium Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from scipy.integrate import solve_ivp

# Predator-prey parameters
r = 1.0        # prey growth rate
K = 10.0       # prey carrying capacity
p = 0.5        # predation rate
d = 0.3        # predator death rate
q = 0.1        # predator efficiency

# Define the system (solve_ivp expects f(t, state))
def system(t, state):
    x, y = state
    dx_dt = r*x*(1 - x/K) - p*x*y
    dy_dt = -d*y + q*x*y
    return [dx_dt, dy_dt]

# Find equilibria numerically
def equations(state):
    x, y = state
    return [
        r*x*(1 - x/K) - p*x*y,
        -d*y + q*x*y
    ]

# Initial guesses for equilibria
eq1 = fsolve(equations, [0.1, 0.1])   # converges to (0, 0)
eq2 = fsolve(equations, [5, 1])       # interior equilibrium

print(f"Equilibrium 1: {eq1}")
print(f"Equilibrium 2: {eq2}")

# Compute Jacobian at equilibrium 2
x_eq, y_eq = eq2
J = np.array([
    [r*(1 - 2*x_eq/K) - p*y_eq, -p*x_eq],
    [q*y_eq, q*x_eq - d]
])

eigenvalues = np.linalg.eigvals(J)
print(f"Jacobian at equilibrium 2:")
print(J)
print(f"Eigenvalues: {eigenvalues}")
print(f"Real parts: {eigenvalues.real}")

# Classify equilibrium
if all(ev.real < 0 for ev in eigenvalues):
    print("Equilibrium is stable")
elif all(ev.real > 0 for ev in eigenvalues):
    print("Equilibrium is unstable")
else:
    print("Equilibrium is a saddle point")

# Simulate and plot phase portrait
x_vals = np.linspace(0.1, 15, 20)
y_vals = np.linspace(0.1, 3, 15)

plt.figure(figsize=(10, 8))

# Direction field
for x0 in x_vals:
    for y0 in y_vals:
        dx, dy = system(0, [x0, y0])
        norm = np.sqrt(dx**2 + dy**2)
        if norm > 0:
            plt.arrow(x0, y0, 0.3*dx/norm, 0.3*dy/norm,
                     head_width=0.1, head_length=0.1, fc='gray', ec='gray')

# Nullclines
x_null = np.linspace(0, 15, 500)
y_null_x = np.linspace(0, 3, 500)

# x-nullcline: x(r(1-x/K) - py) = 0
y_x_null = r*(1 - x_null/K) / p

# y-nullcline: y(-d + qx) = 0 -> x = d/q
x_y_null = d/q * np.ones_like(y_null_x)

plt.plot(x_null, y_x_null, 'b-', linewidth=2, label='x-nullcline')
plt.axvline(d/q, color='r', linewidth=2, label='y-nullcline')

# Plot equilibrium points
plt.plot(0, 0, 'ko', markersize=10, label='Saddle at (0,0)')
plt.plot(x_eq, y_eq, 'go', markersize=10, label=f'Focus at ({x_eq:.2f},{y_eq:.2f})')

# Simulate trajectories
for x0 in [1, 3, 5, 8, 12]:
    for y0 in [0.5, 1.5]:
        t_span = (0, 50)
        sol = solve_ivp(system, t_span, [x0, y0], dense_output=True, max_step=0.1)
        plt.plot(sol.y[0], sol.y[1], 'k-', alpha=0.3, linewidth=0.5)

plt.xlabel('Prey (x)', fontsize=12)
plt.ylabel('Predator (y)', fontsize=12)
plt.title('Phase Portrait: Logistic Predator-Prey System', fontsize=14)
plt.legend(fontsize=11)
plt.xlim(0, 15)
plt.ylim(0, 3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Floquet Theory for Periodic Systems

### The Mathieu Equation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

def mathieu_system(y, t, a, q):
    """
    Convert Mathieu equation to first-order system:
    dy1/dt = y2
    dy2/dt = -(a + 2*q*cos(2*t))*y1
    """
    y1, y2 = y
    return [y2, -(a + 2*q*np.cos(2*t)) * y1]

def monodromy_matrix(a, q, period=np.pi):
    """
    Monodromy matrix M = Phi(T) of the Mathieu equation.
    The coefficient cos(2t) has period T = pi; integrate the two
    columns of the identity matrix over one period.
    """
    t = np.linspace(0, period, 200)
    col1 = odeint(mathieu_system, [1, 0], t, args=(a, q))[-1]
    col2 = odeint(mathieu_system, [0, 1], t, args=(a, q))[-1]
    return np.column_stack((col1, col2))

# Since tr A(t) = 0, Abel's formula gives det M = 1, so the Floquet
# multipliers satisfy lambda1*lambda2 = 1. Solutions remain bounded
# (stable) exactly when |tr M| <= 2, i.e., when both multipliers lie
# on the unit circle.
a_vals = np.linspace(0, 5, 100)
q_vals = np.linspace(0, 2, 50)

stability_map = np.zeros((len(q_vals), len(a_vals)))

for i, q in enumerate(q_vals):
    for j, a in enumerate(a_vals):
        M = monodromy_matrix(a, q)
        stability_map[i, j] = 1 if abs(np.trace(M)) <= 2 else 0

fig, ax = plt.subplots(figsize=(10, 6))
contour = ax.contourf(a_vals, q_vals, stability_map, levels=[0, 0.5, 1],
                      colors=['lightcoral', 'lightgreen'], alpha=0.7)
ax.contour(a_vals, q_vals, stability_map, levels=[0.5], colors='black',
          linewidths=2)

ax.set_xlabel('Parameter $a$', fontsize=11)
ax.set_ylabel('Parameter $q$', fontsize=11)
ax.set_title('Mathieu Equation Stability Diagram\n(Green = Stable, Red = Unstable)',
            fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print stability information for a specific point
a_test, q_test = 1.0, 0.5
M = monodromy_matrix(a_test, q_test)
multipliers = np.abs(np.linalg.eigvals(M))
print(f"Mathieu equation with a={a_test}, q={q_test}:")
print(f"Floquet multipliers |lambda|: {multipliers}")
print(f"Stability: {'Stable' if abs(np.trace(M)) <= 2 else 'Unstable'}")

## Hamiltonian Systems and Symplectic Integrators

### Example: N-Body Gravitational System

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class NBodyHamiltonian:
    """N-body gravitational system using Hamiltonian formulation."""

    def __init__(self, masses, q0, p0, G=1.0):
        """
        masses: array of masses
        q0: initial positions (N x 2 or N x 3)
        p0: initial momenta (N x 2 or N x 3)
        G: gravitational constant
        """
        self.masses = np.array(masses)
        self.N = len(masses)
        self.G = G
        self.q = np.array(q0, dtype=float)
        self.p = np.array(p0, dtype=float)
        self.dim = self.q.shape[1]

    def compute_energy(self):
        """Compute total energy H = T + V."""
        # Kinetic energy
        T = 0.5 * np.sum(self.p**2 / self.masses[:, None])

        # Potential energy
        V = 0.0
        for i in range(self.N):
            for j in range(i+1, self.N):
                r_ij = np.linalg.norm(self.q[i] - self.q[j])
                V -= self.G * self.masses[i] * self.masses[j] / r_ij

        return T + V

    def compute_forces(self):
        """Compute gravitational forces."""
        forces = np.zeros_like(self.q)
        for i in range(self.N):
            for j in range(self.N):
                if i != j:
                    r_ij = self.q[j] - self.q[i]
                    r_mag = np.linalg.norm(r_ij)
                    forces[i] += (self.G * self.masses[i] * self.masses[j] /
                                 r_mag**3) * r_ij

        return forces

    def step_verlet(self, dt):
        """Perform one Verlet symplectic integration step."""
        # Half-step momentum update
        forces = self.compute_forces()
        self.p += 0.5 * dt * forces

        # Full-step position update
        self.q += dt * self.p / self.masses[:, None]

        # Half-step momentum update
        forces = self.compute_forces()
        self.p += 0.5 * dt * forces

    def integrate(self, t_final, dt):
        """Integrate system from t=0 to t=t_final."""
        n_steps = int(t_final / dt)
        energies = [self.compute_energy()]

        for step in range(n_steps):
            self.step_verlet(dt)
            energies.append(self.compute_energy())

        return np.array(energies)

# Two-body orbital example
# Sun-Earth system (scaled units: G = 1, r = 1)
masses = [1.0, 3e-6]   # Sun, Earth
q0 = [[0, 0], [1, 0]]  # Positions
v_circ = np.sqrt(1.0)  # circular-orbit speed sqrt(G*M/r)
p0 = [[0, 0], [0, masses[1] * v_circ]]  # Momenta p = m*v

nbody = NBodyHamiltonian(masses, q0, p0, G=1.0)

# Integrate over ten orbital periods (T = 2*pi in these units)
dt = 0.001
t_final = 10 * 2 * np.pi
energies = nbody.integrate(t_final, dt)
times = np.arange(len(energies)) * dt

# Initial energy
E0 = energies[0]
energy_error = np.abs(energies - E0) / np.abs(E0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(times, energy_error, 'b-', linewidth=1.5, label='Relative Error')
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('|E(t) - E(0)| / |E(0)|', fontsize=11)
ax.set_title('Symplectic Integrator: Energy Conservation', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Initial energy: {E0:.8f}")
print(f"Final energy: {energies[-1]:.8f}")
print(f"Relative error: {energy_error[-1]:.2e}")

## Chaos and Strange Attractors

### Python Implementation: Lorenz and Rössler

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

np.random.seed(42)  # reproducible perturbations

def lorenz(state, t, sigma=10, rho=28, beta=8/3):
    """Lorenz system equations."""
    x, y, z = state
    return [
        sigma * (y - x),
        x * (rho - z) - y,
        x * y - beta * z
    ]

def rossler(state, t, a=0.2, b=0.2, c=5.7):
    """Rossler system equations."""
    x, y, z = state
    return [
        -y - z,
        x + a * y,
        b + z * (x - c)
    ]

def compute_lyapunov_exponent(system, params, initial_state, t_max=1000,
                             n_samples=100):
    """
    Compute largest Lyapunov exponent numerically.
    """
    dt = t_max / n_samples
    lyapunov_sum = 0.0
    epsilon = 1e-8

    # Perturbed initial condition
    state = np.array(initial_state, dtype=float)
    state_pert = state + epsilon * np.random.randn(len(state))

    for _ in range(n_samples):
        t = np.linspace(0, dt, 10)

        # Integrate both trajectories
        traj = odeint(system, state, t, args=params)
        traj_pert = odeint(system, state_pert, t, args=params)

        # Distance between trajectories
        distance = np.linalg.norm(traj[-1] - traj_pert[-1])

        # Update Lyapunov sum
        if distance > 1e-12:
            lyapunov_sum += np.log(distance / epsilon)
            # Re-normalize
            state_pert = traj[-1] + epsilon * (traj_pert[-1] - traj[-1]) / distance

        state = traj[-1]

    return lyapunov_sum / t_max

# Lorenz system
t = np.linspace(0, 50, 10000)
state0 = [1, 1, 1]
trajectory_lorenz = odeint(lorenz, state0, t)
lyap_lorenz = compute_lyapunov_exponent(lorenz, (10, 28, 8/3), state0)

# Rossler system
trajectory_rossler = odeint(rossler, state0, t)
lyap_rossler = compute_lyapunov_exponent(rossler, (0.2, 0.2, 5.7), state0)

# Plot 3D attractors
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(trajectory_lorenz[:, 0], trajectory_lorenz[:, 1],
        trajectory_lorenz[:, 2], 'b-', linewidth=0.5, alpha=0.7)
ax1.set_xlabel('x', fontsize=10)
ax1.set_ylabel('y', fontsize=10)
ax1.set_zlabel('z', fontsize=10)
ax1.set_title(f'Lorenz Attractor (lambda={lyap_lorenz:.3f})',
             fontsize=11)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(trajectory_rossler[:, 0], trajectory_rossler[:, 1],
        trajectory_rossler[:, 2], 'r-', linewidth=0.5, alpha=0.7)
ax2.set_xlabel('x', fontsize=10)
ax2.set_ylabel('y', fontsize=10)
ax2.set_zlabel('z', fontsize=10)
ax2.set_title(f'Rossler Attractor (lambda={lyap_rossler:.3f})',
             fontsize=11)

plt.tight_layout()
plt.show()

print(f"Lorenz system: Largest Lyapunov exponent = {lyap_lorenz:.4f}")
print(f"Rossler system: Largest Lyapunov exponent = {lyap_rossler:.4f}")

### Bifurcation Diagrams for Chaotic Systems

In [ ]:
def bifurcation_diagram_lorenz(rho_range, sigma=10, beta=8/3):
    """
    Create bifurcation diagram for Lorenz system varying rho parameter.
    """
    rho_vals = np.linspace(rho_range[0], rho_range[1], 200)
    t_transient = 500  # Discard transient
    t_measure = 100
    rho_points, z_points = [], []

    for rho in rho_vals:
        state0 = [1, 1, 1]
        # Integrate past transient
        t_trans = np.linspace(0, t_transient, 1000)
        traj_trans = odeint(lorenz, state0, t_trans, args=(sigma, rho, beta))

        # Measure all local maxima of z(t) (the Lorenz map variable)
        t_meas = np.linspace(0, t_measure, 2000)
        state_final = traj_trans[-1]
        traj_meas = odeint(lorenz, state_final, t_meas,
                          args=(sigma, rho, beta))

        z = traj_meas[:, 2]
        peaks = (z[1:-1] > z[:-2]) & (z[1:-1] > z[2:])
        z_peaks = z[1:-1][peaks]
        rho_points.extend([rho] * len(z_peaks))
        z_points.extend(z_peaks)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(rho_points, z_points, 'b.', markersize=1, alpha=0.5)
    ax.set_xlabel('Parameter rho', fontsize=11)
    ax.set_ylabel('Local z-maxima', fontsize=11)
    ax.set_title('Bifurcation Diagram: Lorenz System', fontsize=12)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Generate bifurcation diagram
bifurcation_diagram_lorenz((1, 100))